In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv('../data/cicids2017_cleaned.csv')
df.columns = df.columns.str.strip()
df = df.dropna()
df = df[~df.isin([np.inf, -np.inf]).any(axis=1)].reset_index(drop=True)

In [3]:
y = (df['Label'] != 'BENIGN').astype(int)
_, test_idx = train_test_split(df.index, test_size=0.2, random_state=42, stratify=y)
test_df = df.loc[test_idx].reset_index(drop=True)

In [4]:
print(test_df['Label'].value_counts())

Label
BENIGN              87937
DoS Hulk            46037
DoS GoldenEye        2043
DoS slowloris        1149
DoS Slowhttptest     1113
Heartbleed              3
Name: count, dtype: int64


In [5]:
def save_sample(frame, name):
    frame = frame.sample(frac=1, random_state=1).reset_index(drop=True)  # shuffle
    frame.drop(columns=['Label']).to_csv(f'../demo_samples/{name}.csv', index=False)
    frame[['Label']].to_csv(f'../demo_samples/{name}_ANSWERS.csv', index=False)
    print(f"{name}: {len(frame)} rows")

benign  = test_df[test_df['Label'] == 'BENIGN']
attacks = test_df[test_df['Label'] != 'BENIGN']

In [7]:
save_sample(pd.concat([benign.sample(150, random_state=1),
                       attacks.sample(50, random_state=1)]), 'demo_mixed_200')


save_sample(benign.sample(100, random_state=2), 'demo_benign_only')

for attack in attacks['Label'].unique():
    subset = test_df[test_df['Label'] == attack]
    if len(subset) >= 30:
        safe_name = attack.replace(' ', '_').replace('-', '_')
        save_sample(subset.sample(min(60, len(subset)), random_state=3),
                    f'demo_{safe_name}')

demo_mixed_200: 200 rows
demo_benign_only: 100 rows
demo_DoS_Hulk: 60 rows
demo_DoS_GoldenEye: 60 rows
demo_DoS_Slowhttptest: 60 rows
demo_DoS_slowloris: 60 rows


In [8]:
save_sample(pd.concat([benign.sample(400, random_state=4),
                       attacks.sample(100, random_state=4)]), 'demo_stream_500')

# 5. Broken file to demo the error state (wrong columns)
pd.DataFrame({'foo': [1, 2], 'bar': [3, 4]}).to_csv('../demo_samples/bad_file.csv', index=False)

demo_stream_500: 500 rows


In [9]:
import pickle

scaler = pickle.load(open('../results/scaler.pkl', 'rb'))
xgb    = pickle.load(open('../results/xgboost_tuned_model.pkl', 'rb'))

for name in ['demo_mixed_200', 'demo_benign_only']:
    X = pd.read_csv(f'../demo_samples/{name}.csv')
    X.columns = X.columns.str.strip()
    truth = pd.read_csv(f'../demo_samples/{name}_ANSWERS.csv')['Label']
    pred = xgb.predict(scaler.transform(X))
    actual = (truth != 'BENIGN').astype(int)
    print(f"{name}: predicted {pred.sum()} attacks, actual {actual.sum()}, "
          f"correct {(pred == actual).mean():.1%}")

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- ACK Flag Count
- Active Max
- Active Min
- Active Std
- Average Packet Size
- ...
Feature names seen at fit time, yet now missing:
-  ACK Flag Count
-  Active Max
-  Active Min
-  Active Std
-  Average Packet Size
- ...


In [10]:
X = pd.read_csv('../demo_samples/demo_mixed_200.csv')
print("Scaler expects:", len(scaler.feature_names_in_), "columns")
print("File has:      ", len(X.columns), "columns")
print(list(scaler.feature_names_in_)[:5])
print(list(X.columns)[:5])

Scaler expects: 78 columns
File has:       78 columns
[' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets']
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets']


In [11]:
import pickle

scaler = pickle.load(open('../results/scaler.pkl', 'rb'))
xgb    = pickle.load(open('../results/xgboost_tuned_model.pkl', 'rb'))

# Map stripped name -> the exact name the scaler was fitted with
mapping  = {c.strip(): c for c in scaler.feature_names_in_}
expected = list(scaler.feature_names_in_)

for name in ['demo_mixed_200', 'demo_benign_only']:
    X = pd.read_csv(f'../demo_samples/{name}.csv')
    X = X.rename(columns=lambda c: mapping.get(c.strip(), c))
    X = X[expected]   # same columns, same order as training

    truth  = pd.read_csv(f'../demo_samples/{name}_ANSWERS.csv')['Label']
    pred   = xgb.predict(scaler.transform(X))
    actual = (truth != 'BENIGN').astype(int)
    print(f"{name}: predicted {pred.sum()} attacks, actual {actual.sum()}, "
          f"correct {(pred == actual).mean():.1%}")

c:\Users\Maneshna\miniconda3\envs\torch_env\lib\site-packages\xgboost\core.py:158: UserWarning: [16:04:18] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


demo_mixed_200: predicted 50 attacks, actual 50, correct 100.0%
demo_benign_only: predicted 0 attacks, actual 0, correct 100.0%
